# Example 15.2 / 2: Guidance the skill adds

Companion notebook for **15.2 AMPL Knowledge Skill for AI Agents** in the AMPL lab Documentation, section "Guidance the skill adds". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Guidance the skill adds

`ampl.set_data(products, "PROD")` fills the set from the index and the three parameters from the
columns with the same names.

The skill page lists one rewrite as a typical prompt:
`Rewrite this big-M reformulation using AMPL's MP logical operators instead.` Both forms can be
checked on the same instance. This big-M version links a setup decision `Use` to production:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                  # steel products
param rate {PROD} > 0;     # tons per hour
param profit {PROD};       # dollars per ton
param setup {PROD} >= 0;   # setup cost per product, dollars
param market {PROD} >= 0;  # maximum sales, tons
param avail >= 0;          # rolling hours available

var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
var Use {PROD} binary;                    # 1 if the product is set up

maximize Net_Profit:  # sales profit minus setup costs
    sum {p in PROD} (profit[p] * Make[p] - setup[p] * Use[p]);

# rolling hours used fit within avail
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
subject to Link {p in PROD}: Make[p] <= market[p] * Use[p];  # big-M form: market is the bound

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["setup"] = {"bands": 50000, "coils": 60000}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["Net_Profit"].value())
print({p: round(v) for p, v in ampl.var["Use"].to_dict().items()})

solved 100000.0
{'bands': 1, 'coils': 0}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param:  rate  profit  setup  market :=
bands    200     25   50000    6000
coils    140     30   60000    4000;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display solve_result, Net_Profit;
display Use;

solve_result = solved
Net_Profit = 1e+05

Use [*] :=
bands   1
coils  -2.27374e-16
;



### Another way in amplpy

The rewritten `Link` states the rule directly with the implication operator `==>`. With the same
model.mod and data.py, this run.py drops the big-M `Link` and adds the indicator form in its place:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["setup"] = {"bands": 50000, "coils": 60000}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [11]:
# run.py
ampl.eval("drop Link;")
ampl.eval("subject to Link_If {p in PROD}: Use[p] = 0 ==> Make[p] = 0;")
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["Net_Profit"].value())
print({p: round(v) for p, v in ampl.var["Use"].to_dict().items()})

solved 100000.0
{'bands': 1, 'coils': 0}
